In [1]:
#from dataloader import get_data
from UNet_A_D_AS import T_UNet_Head
import torch
from torch.utils.data import TensorDataset, DataLoader
from torchvision.transforms.functional import resize as torch_resize
from skimage.transform import resize
from keras.utils import load_img, img_to_array
from tqdm.notebook import tqdm
import os
import numpy as np
from torch.utils.data import Dataset, DataLoader

In [2]:
# === Гиперпараметры ===
im_height, im_width = 256, 256  # поменяй под нужный размер
BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-3
DEVICE = torch.device("cuda")

In [3]:
# === DICE LOSS ===
def dice_loss(pred, target, smooth=1.):
    pred = pred.contiguous()
    target = target.contiguous()
    intersection = (pred * target).sum(dim=2).sum(dim=2)
    dice = (2. * intersection + smooth) / (
        pred.sum(dim=2).sum(dim=2) + target.sum(dim=2).sum(dim=2) + smooth
    )
    return 1 - dice.mean()

In [4]:
# === DATA LOADER ===
class TiandituDataset(Dataset):
    def __init__(self, images, masks):
        self.images = images
        self.masks = masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx].transpose(2, 0, 1)  # HWC -> CHW
        mask = self.masks[idx].transpose(2, 0, 1)
        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)

In [5]:
# === ЗАГРУЗКА ДАННЫХ ===
def get_data(path, train=True):
    ids = next(os.walk(path + "images"))[2]
    X = np.zeros((len(ids), im_height, im_width, 1), dtype=np.float32)
    if train:
        y = np.zeros((len(ids), im_height, im_width, 1), dtype=np.float32)

    print('Getting and resizing images ... ')
    for n, id_ in tqdm(enumerate(ids), total=len(ids)):
        img = load_img(path + 'images/' + id_, grayscale=True)
        x_img = img_to_array(img)
        x_img = resize(x_img, (im_height, im_width, 1), mode='constant', preserve_range=True)

        fname, extension = os.path.splitext(id_)
        mask_id_ = fname + '.jpg' if extension != '.jpg' else id_

        if train:
            mask = img_to_array(load_img(path + 'masks/' + mask_id_, grayscale=True))
            mask = resize(mask, (im_height, im_width, 1), mode='constant', preserve_range=True)

        X[n, ..., 0] = x_img.squeeze() / 255
        if train:
            y[n] = mask / 255
    print('Done!')
    return (X, y) if train else X

In [6]:
# === Загрузка numpy-данных ===
X_np, y_np = get_data('../training_dataset/tianditu/', train=True)

Getting and resizing images ... 


  0%|          | 0/1190 [00:00<?, ?it/s]

C:\python3.10\lib\site-packages\keras\utils\image_utils.py:409: UserWarning: grayscale is deprecated. Please use color_mode = "grayscale"
  warnings.warn(


Done!


In [7]:
# === Torch Dataset/DataLoader ===
dataset = TiandituDataset(X_np, y_np)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [8]:
# === Инициализация модели ===
model = T_UNet_Head(in_channels=1).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [9]:
# === ОБУЧЕНИЕ ===
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for images, masks in train_loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        outputs = model(images)
        loss = dice_loss(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.detach().item()

    print(f"Epoch [{epoch+1}/{EPOCHS}]  Loss: {epoch_loss / len(train_loader):.4f}")

Epoch [1/20]  Loss: 0.3630
Epoch [2/20]  Loss: 0.2060
Epoch [3/20]  Loss: 0.1847
Epoch [4/20]  Loss: 0.1723
Epoch [5/20]  Loss: 0.1649
Epoch [6/20]  Loss: 0.1579
Epoch [7/20]  Loss: 0.1496
Epoch [8/20]  Loss: 0.1499
Epoch [9/20]  Loss: 0.1437
Epoch [10/20]  Loss: 0.1453
Epoch [11/20]  Loss: 0.1434
Epoch [12/20]  Loss: 0.1400
Epoch [13/20]  Loss: 0.1319
Epoch [14/20]  Loss: 0.1345
Epoch [15/20]  Loss: 0.1285
Epoch [16/20]  Loss: 0.1292
Epoch [17/20]  Loss: 0.1264
Epoch [18/20]  Loss: 0.1268
Epoch [19/20]  Loss: 0.1280
Epoch [20/20]  Loss: 0.1256
